# Load Library

In [1]:
import json
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import re

# Load Data

In [2]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
test = load_json("/data/alqac25_private_test_task2.json")
law_data = load_json("/data/alqac25_law.json")

# Get content from Article & Law ID

In [4]:
law_map = {}

for law in law_data:
    for article in law.get("articles", []):
        law_map[(law["id"], article["id"])] = article["text"]

In [5]:
def get_law_content(law_infors):
    law_content = ""
    for law_info in law_infors:
        law_id = law_info['law_id']
        article_id = law_info['article_id']
        law_content += f"Luật: {law_id}" + '\n' + law_map.get((law_id, article_id), "") + '\n'*2
    return law_content

# Design prompts for each type of question

## True False Prompt

In [6]:
true_false_prompt = """
Bạn là một chuyên gia pháp lý có nhiệm vụ xác định các câu hỏi bên dưới Đúng hoặc Sai theo luật Việt Nam bằng cách suy luận từng bước.

### Hướng dẫn:
1. Đọc kỹ phần văn bản luật trong mục ### Tài liệu cung cấp.
2. Phân tích câu hỏi trong mục ### Câu hỏi bằng cách suy nghĩ từng bước: 
    - Bước 1: Xác định khái niệm pháp lý,các điều khoản liên quan
    - Bước 2: Áp dụng chúng vào câu hỏi.
3. Sau khi đã suy luận đầy đủ, đưa ra kết luận Đúng hoặc Sai cho câu hỏi.
4. Chỉ sử dụng hai từ để kết luận: **Đúng** hoặc **Sai**.
5. Kết quả đầu ra là một đối tượng JSON CHỈ gồm hai trường như phần ### Mẫu đầu ra:
    - **question**: nội dung câu hỏi
    - **answer**: nội dung trả lời tự luận chi tiết

### Tài liệu cung cấp:
{context}

### Ví dụ: 
{{
    "question": "Theo luật viên chức, một trong các nguyên tắc trong hoạt động nghề nghiệp của viên chức là chịu sự thanh tra, kiểm tra, giám sát của cơ quan, tổ chức có thẩm quyền và của nhân dân, đúng hay sai?",
    "content": "Các nguyên tắc trong hoạt động nghề nghiệp của viên chức\n\n1. Tuân thủ pháp luật, chịu trách nhiệm trước pháp luật trong quá trình thực hiện hoạt động nghề nghiệp.\n\n2. Tận tụy phục vụ nhân dân.\n\n3. Tuân thủ quy trình, quy định chuyên môn, nghiệp vụ, đạo đức nghề nghiệp và quy tắc ứng xử.\n\n4. Chịu sự thanh tra, kiểm tra, giám sát của cơ quan, tổ chức có thẩm quyền và của nhân dân.",
    "answer": "Đúng"
}}
\n
{{
    "question": "Quan hệ hôn nhân và gia đình có yếu tố nước ngoài là quan hệ hôn nhân và gia đình mà ít nhất một bên tham gia là người nước ngoài, người Việt Nam định cư ở nước ngoài, đúng hay sai?",
    "content": "25. Quan hệ hôn nhân và gia đình có yếu tố nước ngoài là quan hệ hôn nhân và gia đình mà ít nhất một bên tham gia là người nước ngoài, người Việt Nam định cư ở nước ngoài; quan hệ hôn nhân và gia đình giữa các bên tham gia là công dân Việt Nam nhưng căn cứ để xác lập, thay đổi, chấm dứt quan hệ đó theo pháp luật nước ngoài, phát sinh tại nước ngoài hoặc tài sản liên quan đến quan hệ đó ở nước ngoài.",
    "answer": "Đúng"
}}
\n
{{
    "question": "Chỉ có một người nắm quyền làm chủ nhà nước, tất cả quyền lực nhà nước tập trung trong tay chủ tịch nước, đúng hay sai?",
    "content": "1. Nhà nước Cộng hòa xã hội chủ nghĩa Việt Nam là nhà nước pháp quyền xã hội chủ nghĩa của Nhân dân, do Nhân dân, vì Nhân dân.\n\n2. Nước Cộng hòa xã hội chủ nghĩa Việt Nam do Nhân dân làm chủ; tất cả quyền lực nhà nước thuộc về Nhân dân mà nền tảng là liên minh giữa giai cấp công nhân với giai cấp nông dân và đội ngũ trí thức.\n\n3. Quyền lực nhà nước là thống nhất, có sự phân công, phối hợp, kiểm soát giữa các cơ quan nhà nước trong việc thực hiện các quyền lập pháp, hành pháp, tư pháp.",
    "answer": "Sai"
}}
\n

### Mẫu đầu ra:
{{
    "question": "Chỉ có một người nắm quyền làm chủ nhà nước, tất cả quyền lực nhà nước tập trung trong tay chủ tịch nước, đúng hay sai?",
    "answer": "Sai"
}}

### Câu hỏi:
{question}

### Kết quả:
"""



## Multiple Choice Prompt

In [7]:
multi_choice_prompt = """
Bạn là một chuyên gia pháp lý có nhiệm vụ chọn đáp án trong các câu hỏi trắc nghiệm theo luật Việt Nam bằng cách suy luận từng bước.

### Hướng dẫn:
1. Đọc kỹ phần văn bản luật trong mục ### Tài liệu cung cấp.
2. Phân tích câu hỏi trong mục ### Câu hỏi bằng cách suy nghĩ từng bước: 
    - Bước 1: Xác định khái niệm pháp lý,các điều khoản liên quan
    - Bước 2: Áp dụng chúng vào câu hỏi.
3. Sau khi đã suy luận đầy đủ, hãy lựa chọn đáp án đúng nhất cho câu hỏi.
4. Chỉ đưa ra đáp án là một trong bốn kí tự: **A**, **B**, **C** hoặc **D**
5. Một số câu hỏi có phương án mang tính tổ hợp như:
    - "Cả A và B đều đúng" (ví dụ phương án C)
    - "Cả A và B đều sai" (ví dụ phương án D)
    - "Tất cả A, B, C đều đúng" (ví dụ phương án D)
    - "Tất cả phương án đều đúng"
    - "Tất cả phương án đều sai"
    Trong trường hợp này, hãy phân tích từng phương án riêng biệt và đối chiếu với văn bản luật, sau đó xác định xem các phương án nào đúng hoặc sai trước khi chọn tổ hợp phù hợp.
6. Kết quả đầu ra là một đối tượng JSON như ví dụ CHỈ gồm hai trường như phần ### Mẫu đầu ra:
    - **question**: nội dung câu hỏi
    - **answer**: nội dung trả lời tự luận chi tiết

### Tài liệu cung cấp:
{context}

### Ví dụ: 
{{
    "question": "Theo Luật Du Lịch, có cách loại cơ sở lưu trú du lịch nào?",
    "content": "Các loại cơ sở lưu trú du lịch\n1. Khách sạn.\n\n2. Biệt thự du lịch.\n\n3. Căn hộ du lịch.\n\n4. Tàu thủy lưu trú du lịch.\n\n5. Nhà nghỉ du lịch.\n\n6. Nhà ở có phòng cho khách du lịch thuê.\n\n7. Bãi cắm trại du lịch.\n\n8. Các cơ sở lưu trú du lịch khác.",
    "choices": {{
        "A": "Khách sạn",
        "B": "Cả 3 đáp án A,B.C đều đúng",
        "C": "Căn hộ du lịch.",
        "D": "Biệt thự du lịch."
    }},
    "answer": "B"
}}
\n
{{
    "question": "Hoạt động của trường quay bao gồm các hoạt động nào sau đây?",
    "content": "Hoạt động của trường quay\n\n1. Tổ chức quản lý, điều hành hoặc hợp tác liên doanh, liên kết sản xuất phim.\n\n2. Cung cấp dịch vụ sản xuất phim và dịch vụ khác theo quy định của pháp luật.",
    "choices": {{
        "A": "Tổ chức quản lý, điều hành hoặc hợp tác liên doanh, liên kết sản xuất phim.",
        "B": "Cung cấp dịch vụ sản xuất phim và dịch vụ khác theo quy định của pháp luật.",
        "C": "Cả A và B đều đúng",
        "D": "Cả A và B đều sai"
    }},
    "answer": "C"
}}
\n
{{
    "question": "Bảo lưu quyền sở hữu KHÔNG chấm dứt trong trường hợp sau đây?",
    "content": "Chấm dứt bảo lưu quyền sở hữu\n\nBảo lưu quyền sở hữu chấm dứt trong trường hợp sau đây:\n\n1. Nghĩa vụ thanh toán cho bên bán được thực hiện xong;\n\n2. Bên bán nhận lại tài sản bảo lưu quyền sở hữu;\n\n3. Theo thỏa thuận của các bên.\n\nTiểu mục 6\n\nBẢO LÃNH",
    "choices": {{
        "A": "Nghĩa vụ thanh toán cho bên bán được thực hiện xong",
        "B": "Bên bán nhận lại tài sản bảo lưu quyền sở hữu",
        "C": "Theo qui định khác trong bộ luật dân sự năm 2015",
        "D": "Theo thỏa thuận của các bên"
    }},
    "answer": "C"
}}
\n

### Mẫu đầu ra:
{{
    "question": "Hội đồng thẩm định, phân loại phim gồm các thành phần nào?",
    "answer": "D"
}}


### Câu hỏi:
{question}
Đáp án lựa chọn:
A. {A}
B. {B}
C. {C}
D. {D}

### Kết quả:
"""



## Free Text Prompt

In [8]:
free_text_prompt = """
Bạn là một chuyên gia pháp lý có nhiệm vụ chọn đáp án trong các câu hỏi trắc nghiệm theo luật Việt Nam bằng cách suy luận từng bước.

### Hướng dẫn:
1. Đọc kỹ phần văn bản luật trong mục ### Tài liệu cung cấp.
2. Phân tích câu hỏi trong mục ### Câu hỏi bằng cách suy nghĩ từng bước:
    - Bước 1: Xác định khái niệm pháp lý hoặc quy định cần tra cứu.
    - Bước 2: Tìm đúng nội dung liên quan trong văn bản luật.
    - Bước 3: Trả lời ngắn gọn, chính xác, không giải thích gì thêm.
3. Tránh suy đoán hoặc trả lời theo cảm tính. Câu trả lời phải bám sát tài liệu cung cấp.
4. Câu trả lời phải ngắn gọn, chính xác vào nội dung câu hỏi, chỉ cần nội dung chính và bỏ qua phần diễn giải.
5. Kết quả đầu ra là một đối tượng JSON chỉ gồm hai trường như phần ### Mẫu đầu ra: 
    - **question**: nội dung câu hỏi
    - **answer**: nội dung trả lời tự luận chi tiết

### Tài liệu cung cấp:
{context}

### Ví dụ:
{{
    "question": "Hồ sơ đề nghị cấp lại thẻ hướng dẫn viên du lịch bao gồm ảnh chân dung màu cỡ bao nhiêu?",
    "content": "Cấp lại thẻ hướng dẫn viên du lịch\n1. Thẻ hướng dẫn viên du lịch được cấp lại trong trường hợp bị mất, bị hư hỏng hoặc thay đổi thông tin trên thẻ. Thời hạn của thẻ hướng dẫn viên du lịch được cấp lại bằng thời hạn còn lại của thẻ đã được cấp.\n\n2. Hồ sơ đề nghị cấp lại thẻ hướng dẫn viên du lịch bao gồm:\n\na) Đơn đề nghị cấp lại thẻ hướng dẫn viên du lịch theo mẫu do Bộ trưởng Bộ Văn hóa, Thể thao và Du lịch quy định;\n\nb) 02 ảnh chân dung màu cỡ 3 cm x 4 cm;\n",
    "answer": "3 cm x 4 cm"
}}
\n
{{
    "question": "Chức năng của Trung tâm trọng tài là gì?",
    "content": "Chức năng của Trung tâm trọng tài\n\nTrung tâm trọng tài có chức năng tổ chức, điều phối hoạt động giải quyết tranh chấp bằng Trọng tài quy chế và hỗ trợ Trọng tài viên về các mặt hành chính, văn phòng và các trợ giúp khác trong quá trình tố tụng trọng tài." ,
    "answer": "Công dân chỉ cần xuất trình thẻ CMND/CCCD, cơ quan có thẩm quyền sẽ tra cứu được các thông tin cư trú trong hệ thống cơ sở dữ liệu quốc gia."
}}
\n
{{
    "question": "Công dân trong độ tuổi nào theo quy định sẽ được gọi là Thanh niên?",
    "content": "Thanh niên\n\nThanh niên là công dân Việt Nam từ đủ 16 tuổi đến 30 tuổi.",
    "answer": "Từ đủ 16 tuổi đến 30 tuổi",
}}

### Mẫu đầu ra:
{{
    "question": "Hồ sơ đề nghị cấp lại thẻ hướng dẫn viên du lịch bao gồm ảnh chân dung màu cỡ bao nhiêu?",
    "answer": "3 cm x 4 cm"
}}

### Câu hỏi:
{question}


### Kết quả:
"""

## Load Prompt & max_new_token

In [17]:
def prompt_design_v2(question):
    # Trích thông tin law_id và article_id (chỉ lấy phần tử đầu tiên trong danh sách relevant_articles)
    law_infors = question['relevant_articles']
    law_content = ""
    for item in  law_infors:
        law_id = item['law_id']
        law_content = get_law_content(law_infors)

    # Xử lý theo từng loại câu hỏi
    if question['question_type'] == 'Đúng/Sai':
        return true_false_prompt.format(context = law_content, question = question["text"]), law_content,128
    elif question['question_type'] == 'Trắc nghiệm':
        choices = question['choices']
        return multi_choice_prompt.format(context = law_content, question = question["text"], A = choices["A"], B = choices["B"], C = choices["C"], D = choices["D"]), law_content, 128

    else: # Tự luận
        return free_text_prompt.format(context = law_content, question = question["text"]), law_content, 250


# Question & Answering

## Get answer from LLMs

In [10]:
def predict_answer(model, tokenizer, question):
    input_text, content, token = prompt_design_v2(question)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    prompt_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            return_dict_in_generate=False,
            output_scores=False,
            max_new_tokens=token,
            do_sample=True,
            temperature=1,
            top_p=1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the generated part (exclude input)
    generated_tokens = outputs[0][prompt_len:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
    return response, content


## Model & Tokenizer Loading

In [11]:
model_name = "AITeamVN/GRPO-VI-Qwen2-7B-RAG"

In [12]:
# Load model and tokenizer
from huggingface_hub import login


print(f"Loading {model_name} model...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    use_cache=True
)

# Set pad token if not exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

Loading AITeamVN/GRPO-VI-Qwen2-7B-RAG model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!
Model device: cuda:0
Model dtype: torch.float16


## Get answer function

In [13]:
import re

def get_answer(data):
    if isinstance(data, list):
        data = ''.join(data)

    # Get query after ### Kết quả
    match = re.search(r'"answer"\s*:\s*"([^"]+)"', data)
    if match:
        return match.group(1)
    else:
        print("Không tìm thấy trường 'ans' sau phần Kết quả.")
        return None

## Answer Prediction and Evaluation Pipeline

In [22]:
def run_pipeline(model, tokenizer):
    results = []
    for question in tqdm(test, desc="Processing and Evaluating questions"):
        max_retry = 10
        retry_count = 0
        print("QuestionID: ", question["question_id"])
        print("Question: ", question["text"])
        while retry_count < max_retry:
            pred_answer, content = predict_answer(model, tokenizer, question)
            print(f"Count {retry_count}")
            print("*"*20)
            print(pred_answer)
            print("*"*20)
            ans = get_answer(pred_answer)
            if ans: 
                break
            retry_count+=1
        print(f"Answer: {ans}")
        results.append({
            "question_id": question["question_id"],
            "question": question["text"],
            "question_type": question["question_type"],
            "content": content,
            "answer": ans
        })
        print("="*30)

    return results

# RUN

# Run the pipeline


In [ ]:
print("Starting evaluation pipeline...")
final_results = run_pipeline(model, tokenizer)
print(f"Total questions processed: {len(final_results)}")

In [24]:
## Filter answer

import re

def extract_answer_letter(text):
    """Get answer (A, B, C, D) from string using regex."""
    match = re.match(r"^\s*([A-D])\.\s+", text)
    return match.group(1) if match else None


In [25]:
for item in final_results:
    if item["question_type"] == "Trắc nghiệm" and len(item["answer"])>2:
        tmp = item["answer"]
        ans = extract_answer_letter(tmp)
        item["answer"] = ans

In [26]:
results = [
    {"question_id": item["question_id"], "answer": item["answer"]}
    for item in final_results
]

In [27]:
output_file = f"{model_name.split('/')[-1]}_private_test_task2.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)